In [ ]:
import sys
import pandapower as pp
import pandapower.networks as nw
import pandas as pd
import nbformat
import matplotlib.pyplot as plt
from pandapower.plotting.plotly import simple_plotly
from pandapower.plotting.plotly import pf_res_plotly
import copy
import pandapower as pp
from pandapower.plotting.plotly import simple_plotly
from pandapower.plotting.plotly import pf_res_plotly


net = pp.networks.create_cigre_network_mv(with_der=False)


net.load['p_mw']= net.load['p_mw'] / 2
net.load['q_mvar']= net.load['q_mvar'] / 2

pp.runpp(net)
# print(net.load)

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


         name  bus      p_mw    q_mvar  const_z_percent  const_i_percent  \
0     Load R1    1  7.497000  1.522331              0.0              0.0   
1     Load R3    3  0.138225  0.034642              0.0              0.0   
2     Load R4    4  0.215825  0.054091              0.0              0.0   
3     Load R5    5  0.363750  0.091164              0.0              0.0   
4     Load R6    6  0.274025  0.068677              0.0              0.0   
5     Load R8    8  0.293425  0.073539              0.0              0.0   
6    Load R10   10  0.237650  0.059561              0.0              0.0   
7    Load R11   11  0.164900  0.041328              0.0              0.0   
8    Load R12   12  7.497000  1.522331              0.0              0.0   
9    Load R14   14  0.104275  0.026134              0.0              0.0   
10   Load CI1    1  2.422500  0.796237              0.0              0.0   
11   Load CI3    3  0.112625  0.069799              0.0              0.0   
12   Load CI

# Step 3


First Justify here the choice of vehicle for each Shop:
(We can choose whatever we want for justification but it needs to be based on something=> Data or assumption)

Mathem:
To do

Postnord:
To do


ICA:

to do

Airmee: 

to do

What to do in Step 3: 
We have the Data from Efleet scheduler: 

Import those Data: Time where the trucks are charging / Energy consumption during the Day

We need to calculate how much we need to charge during the night.

We assume that people can switch the plugs from a car to another at any time.

First Step is NOT Optimizing the Charging (Dumb charging)

Create a dictionary with Different Type of charger and Power rates that we can switch to compare the charging efficiency later : https://alternative-fuels-observatory.ec.europa.eu/general-information/recharging-systems

=> Justify the best choices

Add the car load to the actual power curve


In [7]:
import pandas as pd
from pathlib import Path
import os 

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

notebook_dir = Path(os.getcwd())

def analyze_schedule(file_path: str):
    # Load file
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])
    
    # 1. Consumption per truck
    truck_consumption = df.groupby("VehicleID")["Consumption_kWh"].sum()
    
    # 2. Total consumption (all trucks)
    total_consumption = truck_consumption.sum()
    
    # 3. Count loading time (Location = 1)
    loading_time = df[df["Location"] == 1].groupby(df["date"].dt.date)["date"].count()
    avg_loading_time_per_day = loading_time.mean()
    
    # 4. Required charger power
    charger_power_needed = total_consumption / avg_loading_time_per_day
    
    return {
        "truck_consumption": truck_consumption,
        "total_consumption": total_consumption,
        "avg_loading_time_per_day": avg_loading_time_per_day,
        "charger_power_needed_kW": charger_power_needed
    }

if __name__ == "__main__":
    base_path = notebook_dir.parent / "data" / "Output" 

    shops = ["schedule_Airmee", "schedule_Postnord", "schedule_Mathem", "schedule_ICA"]

    results = {}
    for shop in shops:
        file = base_path / f"{shop}" / f"{shop}.csv"
        print(file)
        results[shop] = analyze_schedule(file)
    
    # Print results
    for shop, res in results.items():
        print(f"\n=== {shop} ===")
        print("Consumption per truck (kWh):")
        print(res["truck_consumption"])
        print(f"Total fleet consumption: {res['total_consumption']:.2f} kWh")
        print(f"Average loading time per day: {res['avg_loading_time_per_day']:.2f} h")
        print(f"Required charger capacity: {res['charger_power_needed_kW']:.2f} kW")


c:\Users\matis\Desktop\KTH\Practical Optimisation\Tutorial 1\Network_optmisation\data\Output\schedule_Airmee\schedule_Airmee.csv
c:\Users\matis\Desktop\KTH\Practical Optimisation\Tutorial 1\Network_optmisation\data\Output\schedule_Postnord\schedule_Postnord.csv
c:\Users\matis\Desktop\KTH\Practical Optimisation\Tutorial 1\Network_optmisation\data\Output\schedule_Mathem\schedule_Mathem.csv
c:\Users\matis\Desktop\KTH\Practical Optimisation\Tutorial 1\Network_optmisation\data\Output\schedule_ICA\schedule_ICA.csv

=== schedule_Airmee ===
Consumption per truck (kWh):
VehicleID
0    334.076245
1    395.804688
2    383.827012
3    499.900371
4    495.936579
5    507.374645
Name: Consumption_kWh, dtype: float64
Total fleet consumption: 2616.92 kWh
Average loading time per day: 69.36 h
Required charger capacity: 37.73 kW

=== schedule_Postnord ===
Consumption per truck (kWh):
VehicleID
0     334.076245
1     395.804688
2     383.827012
3     392.231260
4     390.746813
5     402.092351
6     379

Step 5 Instructions: to minimise the losses we need to have all the lines have the least Current possible (more balanced network)=> Minimise the Current on all the lines. We need to change the scheduler of the charging for this.  (Pandapower has a loss value )=> can also be used 